# Ablation Study — Segmentación sin Preentrenamiento
**Experimento de ablación**: misma arquitectura ResU-Net, mismos datos ASOCA, mismos hiperparámetros.
Única diferencia: encoder con **pesos aleatorios** (sin cargar encoder_best.pth).

Objetivo: comparar contra el modelo con preentrenamiento para medir su contribución real.

In [1]:
# ── Celda 1: Imports ──────────────────────────────────────────────
import os, sys, json
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

sys.path.insert(0, '/media/mrsmile/IA/tesis')
from models.resunet_segmentation import ResUNetSegmentation
from src.dataset_segmentation import CTASegmentationDataset
from src.losses import DiceBCELoss, dice_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 16.6 GB


In [2]:
# ── Celda 2: Hiperparametros ──────────────────────────────────────
# IDENTICOS al entrenamiento original para comparacion justa
EPOCHS      = 300
BATCH_SIZE  = 2
LR_ENCODER  = 1e-4   # mismo LR para ambos grupos (sin pretrain no hay razon de diferenciarlo)
LR_DECODER  = 1e-4
LR_MIN      = 1e-6
PATCH_SIZE  = (128, 128, 128)
PPV         = 6
POS_RATIO   = 0.5
VAL_SPLIT   = 0.2
SEED        = 42

print('Hiperparametros ablation:')
print(f'  Epochs:      {EPOCHS}')
print(f'  Batch size:  {BATCH_SIZE}')
print(f'  LR:          {LR_ENCODER} (encoder y decoder iguales)')
print(f'  Patch size:  {PATCH_SIZE}')
print(f'  Pretrain:    DESACTIVADO -- pesos aleatorios')

Hiperparametros ablation:
  Epochs:      300
  Batch size:  2
  LR:          0.0001 (encoder y decoder iguales)
  Patch size:  (128, 128, 128)
  Pretrain:    DESACTIVADO -- pesos aleatorios


In [3]:
# ── Celda 3: Rutas ────────────────────────────────────────────────
BASE         = '/media/mrsmile/IA/tesis'
MEMMAP_VOL   = os.path.join(BASE, 'data/processed/memmap/volumes')
MEMMAP_META  = os.path.join(BASE, 'data/processed/memmap/meta')
MEMMAP_MASK  = os.path.join(BASE, 'data/processed/memmap/masks')
MEMMAP_MMETA = os.path.join(BASE, 'data/processed/memmap/masks_meta')
SPLIT_JSON   = os.path.join(BASE, 'data/metadata/data_split.json')

with open(SPLIT_JSON) as f:
    split = json.load(f)

all_cases = [
    c.replace('.nii.gz','').replace('.nii','')
    for c in split['asoca']['fine_tuning']
]

np.random.seed(SEED)
idx       = np.random.permutation(len(all_cases))
n_val     = int(len(all_cases) * VAL_SPLIT)
val_ids   = [all_cases[i] for i in idx[:n_val]]
train_ids = [all_cases[i] for i in idx[n_val:]]

print(f'Total casos fine_tuning: {len(all_cases)}')
print(f'Train: {len(train_ids)} | Val: {len(val_ids)}')

Total casos fine_tuning: 30
Train: 24 | Val: 6


In [4]:
# ── Celda 4: Run name y directorios ───────────────────────────────
RUN_NAME = f"seg_ablation_{datetime.now().strftime('%Y%m%d_%H%M')}"
CKPT_DIR = os.path.join(BASE, 'runs/checkpoints/segmentation', RUN_NAME)
TB_DIR   = os.path.join(BASE, 'runs/tensorboard/segmentation', RUN_NAME)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(TB_DIR,   exist_ok=True)

print(f'Run:        {RUN_NAME}')
print(f'Checkpoint: {CKPT_DIR}')
print(f'TensorBoard:{TB_DIR}')
print(f'\nLanzar TensorBoard:')
print(f'tensorboard --logdir {os.path.join(BASE, "runs/tensorboard/segmentation")}')

Run:        seg_ablation_20260309_1217
Checkpoint: /media/mrsmile/IA/tesis/runs/checkpoints/segmentation/seg_ablation_20260309_1217
TensorBoard:/media/mrsmile/IA/tesis/runs/tensorboard/segmentation/seg_ablation_20260309_1217

Lanzar TensorBoard:
tensorboard --logdir /media/mrsmile/IA/tesis/runs/tensorboard/segmentation


In [5]:
# ── Celda 5: Datasets y dataloaders ───────────────────────────────
train_dataset = CTASegmentationDataset(
    vol_dir            = MEMMAP_VOL,
    mask_dir           = MEMMAP_MASK,
    vol_meta_dir       = MEMMAP_META,
    mask_meta_dir      = MEMMAP_MMETA,
    file_list          = train_ids,
    patch_size         = PATCH_SIZE,
    patches_per_volume = PPV,
    positive_ratio     = POS_RATIO,
)

val_dataset = CTASegmentationDataset(
    vol_dir            = MEMMAP_VOL,
    mask_dir           = MEMMAP_MASK,
    vol_meta_dir       = MEMMAP_META,
    mask_meta_dir      = MEMMAP_MMETA,
    file_list          = val_ids,
    patch_size         = PATCH_SIZE,
    patches_per_volume = PPV,
    positive_ratio     = POS_RATIO,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')

✅ Dataset segmentación: 24 volúmenes
Indexando vóxeles coronarios...
✅ Indexación completa
✅ Dataset segmentación: 6 volúmenes
Indexando vóxeles coronarios...
✅ Indexación completa
Train batches: 72
Val batches:   18


In [6]:
# ── Celda 6: Modelo SIN preentrenamiento ─────────────────────────
model = ResUNetSegmentation(in_channels=1, out_channels=1, base=32).to(DEVICE)

# ABLATION: NO se llama load_pretrained_encoder
# Todos los pesos se inicializan con kaiming uniform (PyTorch default)
print('Modelo inicializado con pesos ALEATORIOS (ablation)')
print('load_pretrained_encoder: DESACTIVADO')

total_params   = sum(p.numel() for p in model.parameters())
encoder_params = sum(p.numel() for n,p in model.named_parameters() if n.startswith('e'))
decoder_params = total_params - encoder_params
print(f'\nParametros totales:  {total_params:,}')
print(f'Parametros encoder:  {encoder_params:,}')
print(f'Parametros decoder:  {decoder_params:,}')

criterion = DiceBCELoss()
optimizer = torch.optim.Adam(
    model.get_param_groups(lr_encoder=LR_ENCODER, lr_decoder=LR_DECODER)
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
scaler    = GradScaler()
writer    = SummaryWriter(log_dir=TB_DIR)

print('\nOptimizador, scheduler y scaler listos')

Modelo inicializado con pesos ALEATORIOS (ablation)
load_pretrained_encoder: DESACTIVADO

Parametros totales:  5,686,465
Parametros encoder:  3,556,640
Parametros decoder:  2,129,825

Optimizador, scheduler y scaler listos


/tmp/ipykernel_63512/2591463650.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()


In [7]:
# ── Celda 7: Verificacion batch ───────────────────────────────────
sample_vol, sample_mask = next(iter(train_loader))
print(f'Volumen:  {sample_vol.shape}  dtype={sample_vol.dtype}')
print(f'Mascara:  {sample_mask.shape} dtype={sample_mask.dtype}')
print(f'Positivos en batch: {sample_mask.sum().item():.0f} voxeles '
      f'({100*sample_mask.float().mean().item():.3f}%)')

with torch.no_grad():
    out = model(sample_vol.to(DEVICE))
print(f'Output:   {out.shape}')
print('Verificacion correcta -- arrancando entrenamiento')

Volumen:  torch.Size([2, 1, 128, 128, 128])  dtype=torch.float32
Mascara:  torch.Size([2, 1, 128, 128, 128]) dtype=torch.float32
Positivos en batch: 12415 voxeles (0.296%)
Output:   torch.Size([2, 1, 128, 128, 128])
Verificacion correcta -- arrancando entrenamiento


In [8]:
from src.losses import DiceBCELoss
import inspect
print(inspect.getsource(DiceBCELoss.forward))

    def forward(self, pred, target):
        loss_dice = self.dice(pred, target)
        loss_bce  = self.bce(pred, target)
        return self.dw * loss_dice + self.bw * loss_bce, loss_dice, loss_bce



In [10]:
# ── Celda 8: Loop de entrenamiento ────────────────────────────────
best_dice  = 0.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):

    # ── TRAIN ──────────────────────────────────────────────────────
    model.train()
    train_loss, train_dice = 0.0, 0.0

    pbar = tqdm(train_loader, desc=f'Ep {epoch}/{EPOCHS} train',
                leave=False, ncols=80)
    for vol, mask in pbar:
        vol, mask = vol.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits     = model(vol)
            loss, _, _ = criterion(logits, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        train_dice += dice_score(logits, mask)

    train_loss /= len(train_loader)
    train_dice /= len(train_loader)

    # ── VAL ────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_dice = 0.0, 0.0

    pbar = tqdm(val_loader, desc=f'Ep {epoch}/{EPOCHS} val',
                leave=False, ncols=80)
    with torch.no_grad():
        for vol, mask in pbar:
            vol, mask = vol.to(DEVICE), mask.to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits     = model(vol)
                loss, _, _ = criterion(logits, mask)
            val_loss += loss.item()
            val_dice += dice_score(logits, mask)

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)

    scheduler.step()

    # ── TENSORBOARD ────────────────────────────────────────────────
    writer.add_scalars('Loss', {'train': train_loss, 'val': val_loss}, epoch)
    writer.add_scalars('Dice', {'train': train_dice, 'val': val_dice}, epoch)
    writer.add_scalar('LR/encoder', optimizer.param_groups[0]['lr'], epoch)
    writer.add_scalar('LR/decoder', optimizer.param_groups[1]['lr'], epoch)

    # ── CHECKPOINT ─────────────────────────────────────────────────
    if val_dice > best_dice:
        best_dice  = val_dice
        best_epoch = epoch
        print(f'  Nuevo mejor Dice: {best_dice:.4f}')
        torch.save({
            'epoch':            epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state':  optimizer.state_dict(),
            'best_dice':        best_dice,
            'pretrained':       False,
        }, os.path.join(CKPT_DIR, 'model_best.pth'))

    torch.save({
        'epoch':            epoch,
        'model_state_dict': model.state_dict(),
        'best_dice':        best_dice,
        'pretrained':       False,
    }, os.path.join(CKPT_DIR, f'model_epoch_{epoch:03d}.pth'))

    print(f'Ep {epoch:03d} | '
          f'Loss train {train_loss:.4f} val {val_loss:.4f} | '
          f'Dice train {train_dice:.4f} val {val_dice:.4f} | '
          f'LR {optimizer.param_groups[0]["lr"]:.1e}')

writer.close()
print(f'\nEntrenamiento completado -- Mejor Dice val: {best_dice:.4f} (epoca {best_epoch})')

  Nuevo mejor Dice: 0.0234
Ep 001 | Loss train 1.4038 val 1.1188 | Dice train 0.0816 val 0.0234 | LR 1.0e-04


  Nuevo mejor Dice: 0.4162
Ep 002 | Loss train 0.8347 val 0.6238 | Dice train 0.2367 val 0.4162 | LR 1.0e-04


  Nuevo mejor Dice: 0.5105
Ep 003 | Loss train 0.5583 val 0.5153 | Dice train 0.4681 val 0.5105 | LR 1.0e-04


  Nuevo mejor Dice: 0.5531
Ep 004 | Loss train 0.4563 val 0.4689 | Dice train 0.5717 val 0.5531 | LR 1.0e-04


  Nuevo mejor Dice: 0.6057
Ep 005 | Loss train 0.4984 val 0.4157 | Dice train 0.5201 val 0.6057 | LR 1.0e-04


...

Ep 296 | Loss train 0.2224 val 0.2592 | Dice train 0.8169 val 0.7547 | LR 1.0e-06


Ep 297 | Loss train 0.2234 val 0.2488 | Dice train 0.8054 val 0.7628 | LR 1.0e-06


Ep 298 | Loss train 0.2272 val 0.2787 | Dice train 0.7949 val 0.7359 | LR 1.0e-06


Ep 299 | Loss train 0.2284 val 0.2733 | Dice train 0.8023 val 0.7639 | LR 1.0e-06


Ep 300 | Loss train 0.1904 val 0.2670 | Dice train 0.8314 val 0.7448 | LR 1.0e-06

Entrenamiento completado -- Mejor Dice val: 0.8320 (epoca 278)
